# Combinación de archivos GEE — LST MODIS por municipio (2007–2024)

Lee los 18 CSV anuales `LST_municipios_YYYY.csv` directamente desde el zip y los consolida en un único CSV. Convierte la columna `mean` a grados Celsius aplicando el factor de escala MODIS (`mean × 0.02 − 273.15`).

**Salida:** `data/processed/gee_lst_municipios.csv`

> **Nota de nulos:** El 59.7% de los valores de LST son nulos. Esto es normal en datos MODIS — la nubosidad bloquea el sensor infrarrojo. Los nulos deben tratarse (interpolación, imputación o exclusión) antes del modelado.

In [ ]:
import zipfile, io, os, re, time
import pandas as pd

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
ZIP        = r"C:\Users\nilara\Downloads\google_earth.zip"  # ajustar si es necesario
OUTPUT_CSV = "../data/processed/gee_lst_municipios.csv"
EXCLUDE_YEARS = [2025]

In [ ]:
os.makedirs(os.path.dirname(os.path.abspath(OUTPUT_CSV)), exist_ok=True)

with zipfile.ZipFile(ZIP) as z:
    entries = sorted([n for n in z.namelist() if n.lower().endswith('.csv')])

print(f"Archivos encontrados: {len(entries)}")
for e in entries:
    print(f"  {e}")

In [ ]:
total, first, resumen = 0, True, []
t0 = time.time()

with zipfile.ZipFile(ZIP) as z:
    for entry in entries:
        year_m = re.search(r'\d{4}', os.path.basename(entry))
        if not year_m:
            continue
        year = int(year_m.group())
        if year in EXCLUDE_YEARS:
            print(f"  {year}: omitido")
            continue

        print(f"  {year}...", end="", flush=True)
        t1 = time.time()

        with z.open(entry) as f:
            df = pd.read_csv(f)

        df.insert(0, "source_file", year)
        df['lst_celsius'] = df['mean'] * 0.02 - 273.15
        df = df.drop(columns=['mean'])

        nulos_pct = df['lst_celsius'].isna().mean() * 100
        rows = len(df)
        total += rows
        resumen.append({'año': year, 'filas': rows, 'nulos_%': round(nulos_pct, 1)})

        df.to_csv(OUTPUT_CSV, mode='w' if first else 'a',
                  header=first, index=False, encoding='utf-8')
        first = False

        print(f" {rows:,} filas  nulos: {nulos_pct:.1f}%  ({time.time()-t1:.0f}s)")

print(f"\nTotal: {total:,} filas | {(time.time()-t0)/60:.1f} min")
print(f"CSV guardado en: {os.path.abspath(OUTPUT_CSV)}")
print(f"Tamaño: {os.path.getsize(OUTPUT_CSV)/1_048_576:.0f} MB")

In [ ]:
# Resumen por año
df_res = pd.DataFrame(resumen)
display(df_res.style.format({'filas': '{:,}', 'nulos_%': '{:.1f}%'}))

In [ ]:
# Verificacion rapida
df_check = pd.read_csv(OUTPUT_CSV, nrows=3)
display(df_check)